El nombre de este cuaderno hace referencia a que lo trabajado aquí es **independiente** a la competición de *EXIST 2025*. Vamos a trabajar sobre el perspectivismo del hombre y la mujer:

- Entrenar mejor Transformer por separado (diviendo en hombre y mujer) y comparar el resultado con el mejor Transformer del cuaderno *Transformers.ipynb*.
- Coger el mejor LLM del cuaderno *LLM.ipynb* y a través del prompt le indico si es hombre o mujer.

In [1]:
!pip install -U torchao accelerate transformers

# LLM

En esta sección vamos a trabajar el perspectivismo con el LLM que mejor resultados nos ha dado por el momento, *Mistral*.

A esta técnica también se le conoce en el mundo del NLP como "**Persona Prompting**" (asignación de roles).

In [13]:
import pandas as pd
import numpy as np
import torch
import os
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login
from google.colab import userdata

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuración y rutas

In [14]:
# Modelo Base
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"

# Rutas de salida
DIR_SALIDA = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Perspectivismo_ZeroShot/"
os.makedirs(DIR_SALIDA, exist_ok=True)
CSV_SALIDA_HOMBRE = os.path.join(DIR_SALIDA, "predicciones_instruct_hombre.csv")
CSV_SALIDA_MUJER = os.path.join(DIR_SALIDA, "predicciones_instruct_mujer.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
# Autenticación

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except:
    print("⚠️ No se encontró HF_TOKEN, intentando cargar el modelo públicamente...")

## 2. Definición de los prompts con estructura instruct

Los modelos Instruct de Mistral requieren de las etiquetas [INST] y [/INST]

In [16]:
def prompt_hombre(texto):
    return f"""[INST] Actúa como un hombre. Eres un experto analizando redes sociales.
Clasifica el siguiente texto en una de estas dos categorías: 'Misógino' o 'No misógino'.
Devuelve ÚNICAMENTE la etiqueta correspondiente, sin explicaciones adicionales.
Texto: {texto} [/INST] Etiqueta: """.strip()


In [17]:
def prompt_mujer(texto):
    return f"""[INST] Actúa como una mujer. Eres una experta analizando redes sociales.
Clasifica el siguiente texto en una de estas dos categorías: 'Misógino' o 'No misógino'.
Devuelve ÚNICAMENTE la etiqueta correspondiente, sin explicaciones adicionales.
Texto: {texto} [/INST] Etiqueta: """.strip()

## 3. Carga de datos y modelo (Causal LM)

In [18]:
print("Cargando el dataset de test estático...")
test_df = pd.read_csv(CSV_TEST_TEXT)
if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

y_true = test_df["label"].tolist()

Cargando el dataset de test estático...


In [19]:
print(f"Cargando Tokenizador de {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"

Cargando Tokenizador de mistralai/Mistral-7B-Instruct-v0.3...


In [20]:
print("Cargando Modelo Generativo Base (esto puede tardar un poco)...")
# Cargamos en 16 bits para que quepa en la GPU L4
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.eval()

Cargando Modelo Generativo Base (esto puede tardar un poco)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [21]:
# Pipeline de generación
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=10,      # Queremos respuestas muy cortas
    temperature=0.1,        # Temperatura baja para evitar alucinaciones
    do_sample=True,
    return_full_text=False  # Para que no devuelva el prompt
)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## 4. Función de inferencia

In [22]:
def ejecutar_experimento(prompt_func, nombre_experimento, ruta_salida):
    print(f"\n🚀 Iniciando experimento Zero-Shot: {nombre_experimento}")
    predicciones = []
    y_pred = []
    alucinaciones = 0

    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc=nombre_experimento):
        texto = str(row["text"])
        id_video = row["id_EXIST"]

        prompt = prompt_func(texto)

        with torch.no_grad():
            result = pipe(prompt)

        answer = result[0]['generated_text'].strip().lower()

        # Parseo robusto
        if "no misógino" in answer or "no misogino" in answer or answer.startswith("no"):
            pred_binaria = 0
        elif "misógino" in answer or "misogino" in answer or answer.startswith("mis"):
            pred_binaria = 1
        else:
            pred_binaria = -1 # Alucinación (el modelo contestó otra cosa)
            alucinaciones += 1

        y_pred.append(pred_binaria)

        predicciones.append({
            "id_EXIST": id_video,
            "respuesta_bruta": answer,
            "prediccion_binaria": pred_binaria,
            "label_real": row["label"]
        })

    pd.DataFrame(predicciones).to_csv(ruta_salida, index=False)

    print(f"\n⚠️ Alucinaciones (respuestas no válidas): {alucinaciones} de {len(test_df)}")

    # Evaluar (Scikit-learn penaliza los -1 como fallos, lo cual es correcto)
    f1 = f1_score(y_true, y_pred, average="macro", labels=[0, 1])

    print(f"\nResultados {nombre_experimento}:")
    print(f"F1-Score (Macro): {f1:.4f}")

    # Mostramos la matriz incluyendo la fila/columna de errores si los hay
    labels_matrix = [0, 1] if alucinaciones == 0 else [-1, 0, 1]
    print("Matriz de Confusión (-1 son alucinaciones):")
    print(confusion_matrix(y_true, y_pred, labels=labels_matrix))
    print(classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"], labels=[0, 1]))

    return f1

## 5. Ejecución de los experimentos

In [23]:
f1_hombre = ejecutar_experimento(prompt_hombre, "MISTRAL INSTRUCT - ROL: HOMBRE", CSV_SALIDA_HOMBRE)
f1_mujer = ejecutar_experimento(prompt_mujer, "MISTRAL INSTRUCT - ROL: MUJER", CSV_SALIDA_MUJER)


🚀 Iniciando experimento Zero-Shot: MISTRAL INSTRUCT - ROL: HOMBRE


MISTRAL INSTRUCT - ROL: HOMBRE:   0%|          | 0/502 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
MISTRAL INSTRUCT - ROL: HOMBRE:   2%|▏         | 10/502 [01:22<1:09:38,  8.49s/it][transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=10


⚠️ Alucinaciones (respuestas no válidas): 0 de 502

Resultados MISTRAL INSTRUCT - ROL: HOMBRE:
F1-Score (Macro): 0.5417
Matriz de Confusión (-1 son alucinaciones):
[[187  74]
 [148  93]]
              precision    recall  f1-score   support

 No Misógino       0.56      0.72      0.63       261
    Misógino       0.56      0.39      0.46       241

    accuracy                           0.56       502
   macro avg       0.56      0.55      0.54       502
weighted avg       0.56      0.56      0.55       502


🚀 Iniciando experimento Zero-Shot: MISTRAL INSTRUCT - ROL: MUJER


MISTRAL INSTRUCT - ROL: MUJER: 100%|██████████| 502/502 [1:08:03<00:00,  8.13s/it]


⚠️ Alucinaciones (respuestas no válidas): 0 de 502

Resultados MISTRAL INSTRUCT - ROL: MUJER:
F1-Score (Macro): 0.5655
Matriz de Confusión (-1 son alucinaciones):
[[183  78]
 [135 106]]
              precision    recall  f1-score   support

 No Misógino       0.58      0.70      0.63       261
    Misógino       0.58      0.44      0.50       241

    accuracy                           0.58       502
   macro avg       0.58      0.57      0.57       502
weighted avg       0.58      0.58      0.57       502



In [24]:
print("\n" + "="*50)
print("⚖️ RESUMEN DEL EXPERIMENTO DE PERSPECTIVISMO (ZERO-SHOT)")
print("="*50)
print(f"F1-Score Mistral Rol HOMBRE : {f1_hombre:.4f}")
print(f"F1-Score Mistral Rol MUJER  : {f1_mujer:.4f}")
print("="*50)


⚖️ RESUMEN DEL EXPERIMENTO DE PERSPECTIVISMO (ZERO-SHOT)
F1-Score Mistral Rol HOMBRE : 0.5417
F1-Score Mistral Rol MUJER  : 0.5655
